# COMPASS data availability: Gleason, labs, somatic

Diagnostic notebook (read-only; writes nothing). For the ADT cohort it answers:

1. What fraction of the cohort has **any** Gleason / somatic / lab data at all?
2. Which patients does each pipeline rule exclude, and **why** (no data, data dated after ADT
   start, score documented only in a later note, index on/after the outcome, ...)?
3. What would availability look like if we **predicted from the sequencing report date**
   instead of ADT start (§6)?

Cohort layers:

| Layer | Source | Meaning |
|---|---|---|
| `stage1` | `prostate_adt_survival_cohort_adt.csv` | Every patient with an ADT anchor (Stage 1) |
| `model` | `prediction_inputs_adt<suffix>/aggregated_landmark0.csv` | Landmark-0 modeling cohort after the broad-PSA gate and outcome validity (Stage 3) |

Timing rules being audited, as implemented in `data_preprocessing/build_somatic_gleason_inputs.py`:

* **Indexed analyses** (`gleason`, `sequencing_snv`/`sequencing_any`): pick the observation **closest to
  ADT start** (either side; an exact tie goes to the earlier date), then re-base follow-up from that
  observation date. A patient is dropped if the selected date is on/after the event or last contact.
* **Available-case analyses** (`gleason_available_case`, `gleason_somatic_available_case`): keep the
  ADT clock. Gleason must be *available* by ADT start, where available = `max(gleason_date,
  source_note_date)`. Somatic uses the latest specimen with `SAMPLE_COLLECTION_DT <= ADT start`;
  `REPORT_DT` is **not** checked, which §4b quantifies.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

COMPASS_DIR = Path.cwd() if (Path.cwd() / "data_preprocessing").exists() else Path.cwd().parent
sys.path.insert(0, str(COMPASS_DIR / "data_preprocessing"))
import build_prediction_inputs as bpi  # noqa: E402
import build_somatic_gleason_inputs as sg  # noqa: E402

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ID = sg.ca.ID_COL
ENDPOINT = "platinum"  # or "nepc"
EVENT_COL, EVENT_DATE_COL, DURATION_COL = {
    "platinum": ("PLATINUM", "PLATINUM_DATE", "t_platinum"),
    "nepc": ("NEPC", "NEPC_DATE", "t_nepc"),
}[ENDPOINT]

# Same output root as compass_pipeline._PROFILE_OUTPUT_ROOT.
DATA_ROOT = Path(os.environ.get("COMPASS_DATA_ROOT", "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/"))
SUFFIX = "" if ENDPOINT == "platinum" else f"_{ENDPOINT}"
INPUTS_DIR = DATA_ROOT / "survival_analysis" / f"prediction_inputs_adt{SUFFIX}"
SURVIVAL_COHORT_PATH = DATA_ROOT / "prostate_adt_survival_cohort_adt.csv"
LONGITUDINAL_PATH = DATA_ROOT / "longitudinal_prediction_data_adt.csv"
BASE_PATH = INPUTS_DIR / bpi.aggregated_filename(0)
BUILT_SG_DIR = INPUTS_DIR / "somatic_gleason"
SOMATIC_PATH = sg.DEFAULT_SOMATIC_PATH
GLEASON_PATH = sg.DEFAULT_GLEASON_PATH

MIN_PSA_COUNT = bpi.DEFAULT_MIN_PSA_COUNT
PSA_LAB, TESTOSTERONE_LAB = "PSA", "Testosterone"
HIST_CLIP_DAYS = 5 * 365
BAR_COLOR, REF_COLOR = "#2a78d6", "#52514e"

for path in (SURVIVAL_COHORT_PATH, LONGITUDINAL_PATH, BASE_PATH, SOMATIC_PATH, GLEASON_PATH):
    print(f"{'ok     ' if path.exists() else 'MISSING'} {path}")

In [ ]:
def to_mrn(frame):
    out = frame.copy()
    out[ID] = pd.to_numeric(out[ID], errors="coerce")
    out = out.loc[out[ID].notna()].copy()
    out[ID] = out[ID].astype(int)
    return out


def pct(n, d):
    return f"{n:,}/{d:,} ({100 * n / d:.1f}%)" if d else f"{n:,}/0"


def flag_table(flags: pd.DataFrame) -> pd.DataFrame:
    # One row per boolean column: n and % of the layer.
    d = len(flags)
    return pd.DataFrame(
        {"n": flags.sum().astype(int), "pct": (100 * flags.mean()).round(1)}
    ).rename_axis(f"of {d:,} patients")


def reason_table(reasons: pd.Series, order=None) -> pd.DataFrame:
    counts = reasons.value_counts()
    if order is not None:
        counts = counts.reindex([o for o in order if o in counts.index] +
                                [o for o in counts.index if o not in order])
    return pd.DataFrame({"n": counts, "pct": (100 * counts / len(reasons)).round(1)}).rename_axis(
        f"of {len(reasons):,} patients"
    )


def ladder(index, rules):
    # First failing rule wins; rules is an ordered list of (label, bool Series).
    out = pd.Series(pd.NA, index=index, dtype="object")
    for label, failed in rules:
        failed = failed.reindex(index, fill_value=False).astype(bool)
        out = out.mask(out.isna() & failed, label)
    return out


def days_summary(days: pd.Series) -> pd.Series:
    days = days.dropna()
    return pd.Series({
        "n": len(days),
        "before (<0)": f"{(days < 0).mean():.1%}",
        "same day": f"{(days == 0).mean():.1%}",
        "after (>0)": f"{(days > 0).mean():.1%}",
        "median": days.median(),
        "IQR": f"[{days.quantile(.25):.0f}, {days.quantile(.75):.0f}]",
        "|d| <= 90": f"{(days.abs() <= 90).mean():.1%}",
        "|d| <= 365": f"{(days.abs() <= 365).mean():.1%}",
    })


def timing_hist(ax, days, title, xlabel, ref_label):
    days = pd.Series(days).dropna()
    clipped = days.clip(-HIST_CLIP_DAYS, HIST_CLIP_DAYS)
    ax.hist(clipped, bins=60, color=BAR_COLOR, edgecolor="white", linewidth=0.5)
    ax.axvline(0, color=REF_COLOR, linestyle="--", linewidth=1)
    ax.text(0, ax.get_ylim()[1] * 0.95, f" {ref_label}", color=REF_COLOR, va="top", fontsize=8)
    ax.set_title(f"{title}\n(n={len(days):,}; clipped to ±{HIST_CLIP_DAYS // 365}y)", fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("patients")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

## 0. Load sources

In [ ]:
surv = to_mrn(pd.read_csv(SURVIVAL_COHORT_PATH, low_memory=False))
for col in ("TREATMENT_ANCHOR_DATE", "LAST_CONTACT_DATE", EVENT_DATE_COL):
    surv[col] = pd.to_datetime(surv[col], errors="coerce")
n_dup = int(surv[ID].duplicated().sum())
surv = surv.drop_duplicates(ID).set_index(ID)
surv = surv.loc[surv["TREATMENT_ANCHOR_DATE"].notna()]
print(f"Stage-1 ADT cohort: {len(surv):,} patients ({n_dup} duplicate MRN rows dropped)")

want = [ID, "TREATMENT_ANCHOR_DATE", "LAB_DATE", "t_lab", "LAB_NAME", "RAW_TEST_CODE",
        "FIRST_RECORD_DATE", "AGE_AT_TREATMENTSTART", "t_last_contact", "t_death", DURATION_COL]
header = pd.read_csv(LONGITUDINAL_PATH, nrows=0).columns
long = to_mrn(pd.read_csv(LONGITUDINAL_PATH, usecols=[c for c in want if c in header], low_memory=False))
long["LAB_DATE"] = pd.to_datetime(long["LAB_DATE"], errors="coerce")
labs = long.loc[long["LAB_NAME"].notna() & long["LAB_DATE"].notna(), [ID, "LAB_DATE", "LAB_NAME"]]
patient_cols = [c for c in want[6:] if c in long.columns]
pat = long.groupby(ID)[patient_cols].first()
print(f"Longitudinal lab file: {len(long):,} rows, {len(pat):,} patients, {len(labs):,} dated lab rows")

base = to_mrn(pd.read_csv(BASE_PATH, low_memory=False))
model_ids = pd.Index(base[ID].unique(), name=ID)
print(f"Landmark-0 modeling cohort ({ENDPOINT}): {len(model_ids):,} patients, "
      f"{int(base[EVENT_COL].sum()):,} events")

anchors = sg.load_treatment_anchors(LONGITUDINAL_PATH)
shared = surv.index.intersection(anchors.index)
anchor_mismatch = int((anchors[shared] != surv.loc[shared, "TREATMENT_ANCHOR_DATE"]).sum())
print(f"ADT anchors from longitudinal file: {len(anchors):,}; "
      f"disagree with Stage-1 TREATMENT_ANCHOR_DATE for {anchor_mismatch:,} of {len(shared):,} shared patients")
stage1_anchor = surv["TREATMENT_ANCHOR_DATE"]

In [ ]:
gleason_raw = to_mrn(pd.read_parquet(GLEASON_PATH))
gleason = sg.load_gleason(GLEASON_PATH)
print(f"Gleason timeline: {len(gleason):,} rows, {gleason[ID].nunique():,} patients; "
      f"null gleason_date={gleason['gleason_date'].isna().sum():,}, "
      f"null score={gleason[sg.GLEASON_FEATURE].isna().sum():,}, "
      f"documented later than scored={(gleason[sg.GLEASON_AVAILABLE_DATE] > gleason['gleason_date']).sum():,}")

somatic_cols = [c for c in (ID, "SAMPLE_ACCESSION_NBR", "TEST_TYPE", "PANEL_VERSION",
                            "SAMPLE_COLLECTION_DT", "TEST_ORDER_DT", "REPORT_DT")
                if c in pq.read_schema(SOMATIC_PATH).names]
somatic = to_mrn(pd.read_parquet(SOMATIC_PATH, columns=somatic_cols))
for col in ("SAMPLE_COLLECTION_DT", "TEST_ORDER_DT", "REPORT_DT"):
    if col in somatic.columns:
        somatic[col] = pd.to_datetime(somatic[col], errors="coerce")
somatic[sg.SEQUENCING_DATE] = somatic["SAMPLE_COLLECTION_DT"]
somatic[sg.SOMATIC_AVAILABLE_DATE] = somatic["SAMPLE_COLLECTION_DT"]
somatic["_specimen"] = 1  # stand-in feature so the pipeline selectors can run without the wide matrix
lag = (somatic["REPORT_DT"] - somatic["SAMPLE_COLLECTION_DT"]).dt.days
print(f"Somatic testing groups: {len(somatic):,} rows, {somatic[ID].nunique():,} patients; "
      f"null collection={somatic['SAMPLE_COLLECTION_DT'].isna().sum():,}, "
      f"null report={somatic['REPORT_DT'].isna().sum():,}")
print("collection -> report lag (days):", lag.describe(percentiles=[.05, .5, .95]).round(0).to_dict())

## 1. Cohort funnel and lab availability

Why a Stage-1 ADT patient is **not** in the landmark-0 modeling cohort. The first failing rule wins,
in the order `build_prediction_inputs.py` applies them: broad-PSA prevalence gate (>= `MIN_PSA_COUNT`
PSA results of any assay, at any time), then the `make_outcome_df` validity gate, then missing age.
An endpoint at or before ADT start is excluded **by design** (incident endpoint only).

In [ ]:
p = pat.reindex(surv.index)
psa_counts = bpi.load_broad_psa_counts(long, ID).reindex(surv.index, fill_value=0)
in_long = surv.index.isin(pat.index)
in_model = pd.Series(surv.index.isin(model_ids), index=surv.index)

funnel_rules = [
    ("no rows in longitudinal lab file", pd.Series(~in_long, index=surv.index)),
    (f"broad PSA count < {MIN_PSA_COUNT}", psa_counts < MIN_PSA_COUNT),
    ("missing first-record / death / last-contact time",
     p["FIRST_RECORD_DATE"].isna() | p["t_death"].isna() | p["t_last_contact"].isna()),
    ("death or last contact on/before ADT start", (p["t_death"] <= 0) | (p["t_last_contact"] <= 0)),
    (f"{ENDPOINT} missing or on/before ADT start ({DURATION_COL} <= 0)",
     ~(p[DURATION_COL] > 0) if DURATION_COL in p.columns else pd.Series(True, index=surv.index)),
    ("missing age at ADT start", p["AGE_AT_TREATMENTSTART"].isna()),
]
funnel_reason = ladder(surv.index, funnel_rules)
unexplained_out = funnel_reason.isna() & ~in_model
explained_in = funnel_reason.notna() & in_model
funnel_reason = funnel_reason.where(~in_model, "in modeling cohort").fillna("other (not reproduced here)")
display(reason_table(funnel_reason, ["in modeling cohort"] + [r for r, _ in funnel_rules]))
print(f"Patients in the modeling cohort but failing a rule here: {int(explained_in.sum())} "
      f"(should be 0); excluded with no reproduced reason: {int(unexplained_out.sum())}")
print(f"Modeling-cohort MRNs absent from Stage 1: {len(model_ids.difference(surv.index))}")

In [ ]:
def pre_index_lab_flags(index_dates: pd.Series) -> pd.DataFrame:
    # Labs strictly before the index date (matches t_lab < landmark at landmark 0).
    idx = index_dates.dropna().rename("_index")
    m = labs.merge(idx, left_on=ID, right_index=True, how="inner")
    m = m.loc[m["LAB_DATE"] < m["_index"]]
    out = pd.DataFrame(index=index_dates.index)
    out["any lab before index"] = out.index.isin(m[ID])
    out["PSA before index"] = out.index.isin(m.loc[m["LAB_NAME"].eq(PSA_LAB), ID])
    out["testosterone before index"] = out.index.isin(m.loc[m["LAB_NAME"].eq(TESTOSTERONE_LAB), ID])
    out["n labs before index"] = m.groupby(ID).size().reindex(out.index, fill_value=0)
    return out


def lab_availability(ids: pd.Index) -> pd.DataFrame:
    flags = pre_index_lab_flags(stage1_anchor.reindex(ids))
    flags.insert(0, "any lab (any time)", ids.isin(labs[ID]))
    flags.insert(1, f"broad PSA >= {MIN_PSA_COUNT} (any time)", psa_counts.reindex(ids, fill_value=0).values >= MIN_PSA_COUNT)
    return flags


lab_layers = {"stage1": lab_availability(surv.index), "model": lab_availability(model_ids)}
display(pd.concat({k: flag_table(v.drop(columns="n labs before index")) for k, v in lab_layers.items()}, axis=1))
for k, v in lab_layers.items():
    n = v.loc[v["any lab before index"], "n labs before index"]
    print(f"{k}: pre-ADT lab count among patients with any: median={n.median():.0f}, IQR=[{n.quantile(.25):.0f}, {n.quantile(.75):.0f}]")

lab_feature_cols = [c for c in base.columns if "__" in c]
all_nan = base[lab_feature_cols].isna().all(axis=1)
print(f"Modeling cohort rows with all-NaN lab summary features (kept, mean-imputed): {pct(int(all_nan.sum()), len(base))}")

## 2. Any Gleason / any somatic data (no timing restriction)

In [ ]:
gleason_dated_ids = gleason.loc[gleason["gleason_date"].notna() & gleason[sg.GLEASON_FEATURE].notna(), ID]
somatic_dated_ids = somatic.loc[somatic["SAMPLE_COLLECTION_DT"].notna(), ID]
somatic_reported_ids = somatic.loc[somatic["REPORT_DT"].notna(), ID]


def any_data_flags(ids: pd.Index) -> pd.DataFrame:
    out = pd.DataFrame(index=ids)
    out["any Gleason row"] = ids.isin(gleason_raw[ID])
    out["dated Gleason with score"] = ids.isin(gleason_dated_ids)
    out["any somatic row"] = ids.isin(somatic[ID])
    out["somatic with collection date"] = ids.isin(somatic_dated_ids)
    out["somatic with report date"] = ids.isin(somatic_reported_ids)
    out["Gleason AND somatic"] = out["dated Gleason with score"] & out["somatic with collection date"]
    out["neither"] = ~out["dated Gleason with score"] & ~out["somatic with collection date"]
    return out


any_layers = {"stage1": any_data_flags(surv.index), "model": any_data_flags(model_ids)}
display(pd.concat({k: flag_table(v) for k, v in any_layers.items()}, axis=1))

# Are Gleason/somatic patients disproportionately lost between Stage 1 and the modeling cohort?
s1 = any_layers["stage1"]
xt = pd.DataFrame(index=reason_table(funnel_reason, ["in modeling cohort"] + [r for r, _ in funnel_rules]).index, data={
    "all Stage-1": funnel_reason.value_counts(),
    "has Gleason": funnel_reason[s1["dated Gleason with score"]].value_counts(),
    "has somatic": funnel_reason[s1["somatic with collection date"]].value_counts(),
    "has both": funnel_reason[s1["Gleason AND somatic"]].value_counts(),
}).fillna(0).astype(int)
display(xt.astype(str) + " (" + (100 * xt / xt.sum()).round(1).astype(str) + "%)")

## 3. Gleason under the ADT-start design

### 3a. Indexed `gleason` analysis (score closest to ADT start, follow-up from the score date)

Built here with the pipeline's own `build_indexed_feature_sets`, then each excluded patient gets a
reason. Note that `closest_observation_to_adt` selects on date only, so a selected row with a missing
score is **kept** with `GLEASON_SCORE = NaN`.

In [ ]:
base_dates = sg._materialize_absolute_followup_dates(base, anchors)
indexed = sg.build_indexed_feature_sets(
    base, {"sequencing_snv": (somatic[[ID, sg.SEQUENCING_DATE, sg.SOMATIC_AVAILABLE_DATE]], [])},
    gleason, treatment_anchors=anchors, endpoint=ENDPOINT,
)
event = pd.to_numeric(base_dates[EVENT_COL], errors="coerce").fillna(0).eq(1)
followup_end = pd.Series(
    base_dates["LAST_CONTACT_DATE"].where(~event, base_dates[EVENT_DATE_COL]).values, index=base_dates[ID]
)
model_anchor = anchors.reindex(model_ids)


def indexed_reasons(frame, date_col, built, value_col=None):
    selected = sg.closest_observation_to_adt(
        frame, anchors.reindex(model_ids), date_col=date_col,
        value_cols=[value_col] if value_col else [], combine_date_ties_with_max=value_col is None,
    )
    built_ids = pd.Index(built[ID])
    has_rows = pd.Series(model_ids.isin(frame[ID]), index=model_ids)
    has_dated = pd.Series(model_ids.isin(frame.loc[frame[date_col].notna(), ID]), index=model_ids)
    sel_dur = (followup_end.reindex(model_ids) - selected[sg.INDEX_DATE].reindex(model_ids)).dt.days
    rules = [
        ("no rows", ~has_rows),
        (f"no row with a {date_col}", ~has_dated),
        (f"selected date on/after {ENDPOINT} or last contact", ~(sel_dur > 0)),
    ]
    reasons = ladder(model_ids, rules)
    if value_col:
        nan_score = selected[value_col].reindex(model_ids).isna()
        reasons = reasons.mask(reasons.isna() & nan_score, "included, selected score missing")
    reasons = reasons.fillna("included")
    mismatch = (reasons.str.startswith("included") != model_ids.isin(built_ids)).sum()
    print(f"built cohort n={len(built_ids):,}; reason/built disagreement: {mismatch} (should be 0)")
    return reasons, selected


gleason_idx_reason, gleason_selected = indexed_reasons(
    gleason, "gleason_date", indexed["gleason"], value_col=sg.GLEASON_FEATURE
)
display(reason_table(gleason_idx_reason))
display(days_summary(gleason_selected[sg.INDEX_TO_ADT_DAYS]).to_frame("selected Gleason date − ADT start (days)"))

### 3b. Gleason available by ADT start (available-case design, ADT clock kept)

Same status ladder is reused in §6 with the report date as the index.

In [ ]:
def gleason_status_at(index_dates: pd.Series) -> pd.Series:
    # Mirrors build_available_case_sensitivity(analysis="gleason"): closest documented row to the
    # index, then drop it if its score is missing (no fallback to another documented row).
    ids = index_dates.index
    idx = index_dates.dropna()
    selected = sg.closest_available_by_landmark(
        gleason, pd.DataFrame({ID: idx.index, "_landmark_date": idx.values}),
        date_col="gleason_date", available_date_col=sg.GLEASON_AVAILABLE_DATE,
        value_cols=[sg.GLEASON_FEATURE],
    )
    g = gleason.merge(idx.rename("_index"), left_on=ID, right_index=True, how="inner")
    scored_by = set(g.loc[g["gleason_date"].le(g["_index"]), ID])
    status = ladder(ids, [
        ("no Gleason rows", pd.Series(~ids.isin(gleason[ID]), index=ids)),
        ("no dated Gleason row", pd.Series(~ids.isin(gleason.loc[gleason["gleason_date"].notna(), ID]), index=ids)),
    ])
    sel_valid = ids.isin(selected.index[selected[sg.GLEASON_FEATURE].notna()])
    status = status.mask(status.isna() & sel_valid, "available by index")
    status = status.mask(status.isna() & ids.isin(selected.index), "documented by index, score missing")
    status = status.mask(status.isna() & ids.isin(list(scored_by)), "scored by index, documented only after index")
    return status.fillna("all scores dated after index")


GLEASON_ORDER = ["available by index", "documented by index, score missing",
                 "scored by index, documented only after index", "all scores dated after index",
                 "no dated Gleason row", "no Gleason rows"]
gleason_ac_status = gleason_status_at(model_anchor)
display(reason_table(gleason_ac_status, GLEASON_ORDER))

## 4. Somatic under the ADT-start design

### 4a. Indexed `sequencing_snv` / `sequencing_any` (specimen closest to ADT start, follow-up from collection)

Both encodings select the same specimen, so they share one cohort.

In [ ]:
seq_idx_reason, seq_selected = indexed_reasons(somatic, sg.SEQUENCING_DATE, indexed["sequencing_snv"])
display(reason_table(seq_idx_reason))

sel = seq_selected[[sg.INDEX_DATE]].join(model_anchor.rename("adt")).reset_index()
sel = sel.merge(somatic[[ID, "SAMPLE_COLLECTION_DT", "REPORT_DT"]],
                left_on=[ID, sg.INDEX_DATE], right_on=[ID, "SAMPLE_COLLECTION_DT"], how="left")
sel = sel.groupby(ID).agg(adt=("adt", "first"), collected=("SAMPLE_COLLECTION_DT", "first"),
                          reported=("REPORT_DT", "min"))
display(pd.concat({
    "collection − ADT (days)": days_summary((sel["collected"] - sel["adt"]).dt.days),
    "report − ADT (days)": days_summary((sel["reported"] - sel["adt"]).dt.days),
}, axis=1))

### 4b. Somatic available by ADT start (available-case design)

The pipeline treats a specimen as available on its **collection** date. A specimen collected before
ADT but reported after ADT start was not actually known at ADT start; that row shows how many
available-case patients depend on such specimens.

In [ ]:
def somatic_status_at(index_dates: pd.Series) -> pd.Series:
    idx = index_dates.dropna().rename("_index")
    s = somatic.merge(idx, left_on=ID, right_index=True, how="inner")
    collected_by = s["SAMPLE_COLLECTION_DT"].le(s["_index"])
    reported_by = collected_by & s["REPORT_DT"].le(s["_index"])
    no_report = collected_by & s["REPORT_DT"].isna()
    in_ids = lambda mask: index_dates.index.isin(s.loc[mask, ID])
    status = ladder(index_dates.index, [
        ("no somatic rows", pd.Series(~index_dates.index.isin(somatic[ID]), index=index_dates.index)),
        ("no collection date", pd.Series(~index_dates.index.isin(somatic_dated_ids), index=index_dates.index)),
    ])
    status = status.mask(status.isna() & in_ids(reported_by), "collected and reported by index")
    status = status.mask(status.isna() & in_ids(no_report), "collected by index, report date missing")
    status = status.mask(status.isna() & in_ids(collected_by), "collected by index, reported only after index")
    return status.fillna("all specimens collected after index")


SOMATIC_ORDER = ["collected and reported by index", "collected by index, report date missing",
                 "collected by index, reported only after index", "all specimens collected after index",
                 "no collection date", "no somatic rows"]
SOMATIC_PIPELINE_AVAILABLE = SOMATIC_ORDER[:3]  # collection-date rule used by the builder
somatic_ac_status = somatic_status_at(model_anchor)
display(reason_table(somatic_ac_status, SOMATIC_ORDER))

### 4c. Gleason ∩ somatic available case, and the timing figure

In [ ]:
g_ok = gleason_ac_status.eq("available by index")
s_ok = somatic_ac_status.isin(SOMATIC_PIPELINE_AVAILABLE)
s_ok_report = somatic_ac_status.eq("collected and reported by index")
both = pd.DataFrame({
    "Gleason available": g_ok,
    "somatic collected by ADT (pipeline rule)": s_ok,
    "somatic reported by ADT (strict rule)": s_ok_report,
    "Gleason ∩ somatic (pipeline rule)": g_ok & s_ok,
    "Gleason ∩ somatic (strict rule)": g_ok & s_ok_report,
})
display(flag_table(both))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
timing_hist(axes[0, 0], gleason_selected[sg.INDEX_TO_ADT_DAYS],
            "Gleason closest to ADT (3a)", "Gleason date − ADT start (days)", "ADT start")
timing_hist(axes[0, 1], (sel["collected"] - sel["adt"]).dt.days,
            "Specimen closest to ADT (4a)", "collection − ADT start (days)", "ADT start")
timing_hist(axes[1, 0], (sel["reported"] - sel["adt"]).dt.days,
            "Report date of that specimen", "report − ADT start (days)", "ADT start")
first_report = somatic.dropna(subset=["REPORT_DT"]).groupby(ID)["REPORT_DT"].min()
timing_hist(axes[1, 1], (first_report.reindex(surv.index) - stage1_anchor).dt.days,
            "First sequencing report (Stage-1 cohort)", "first report − ADT start (days)", "ADT start")
fig.tight_layout()
plt.show()

## 5. Reconcile with the built `somatic_gleason/` outputs

Compares the cohorts recomputed above with what Stage 3b last wrote to disk. Disagreement means the
built inputs are stale relative to the current sources (or the rules changed).

In [ ]:
def available_case_ids(analysis):
    out = sg.build_available_case_sensitivity(
        base, somatic, ["_specimen"], gleason, treatment_anchors=anchors, landmark_day=0, analysis=analysis,
    )
    return pd.Index(out[ID])


recomputed = {
    "gleason": pd.Index(indexed["gleason"][ID]),
    "sequencing_snv": pd.Index(indexed["sequencing_snv"][ID]),
    "sequencing_any": pd.Index(indexed["sequencing_snv"][ID]),
    "gleason_available_case": available_case_ids("gleason"),
    sg.GLEASON_SOMATIC_AVAILABLE_CASE: available_case_ids("gleason_somatic"),
}
# The §3b/§4c reason ladders must reproduce the pipeline selectors exactly.
for name, ladder_ids in (("gleason_available_case", model_ids[g_ok.values]),
                         (sg.GLEASON_SOMATIC_AVAILABLE_CASE, model_ids[(g_ok & s_ok).values])):
    print(f"{name}: ladder vs pipeline symmetric difference = "
          f"{len(ladder_ids.symmetric_difference(recomputed[name]))} (should be 0)")
rows = []
for name, ids in recomputed.items():
    path = BUILT_SG_DIR / name / bpi.aggregated_filename(0)
    if not path.exists():
        rows.append({"analysis": name, "recomputed": len(ids), "built": None, "only recomputed": None, "only built": None})
        continue
    built_ids = pd.Index(to_mrn(pd.read_csv(path, usecols=[ID]))[ID])
    rows.append({"analysis": name, "recomputed": len(ids), "built": len(built_ids),
                 "only recomputed": len(ids.difference(built_ids)), "only built": len(built_ids.difference(ids))})
display(pd.DataFrame(rows).set_index("analysis"))

## 6. Alternative design: predict from the sequencing report date

Index = a sequencing `REPORT_DT` instead of ADT start. Base = every Stage-1 ADT patient, since the
ADT-anchored gates (broad PSA, endpoint before ADT) no longer define the cohort; the modeling-cohort
subset is shown for comparison. Follow-up uses Stage-1 `LAST_CONTACT_DATE` / event date. A patient is
dropped if the event or last contact falls on/before the report date.

Three ways to choose the report:

* `first_report`: earliest report.
* `first_report_on_or_after_adt`: earliest report on/after ADT start (sequenced on ADT).
* `closest_report_to_adt`: report closest to ADT start, either side (an exact tie goes to the earlier date), mirroring §4a.

At the index, somatic data is available by construction (the reported specimen). Gleason uses the
same documented-by-index rule as §3b; labs are those strictly before the report date.

In [ ]:
reports = somatic.loc[somatic["REPORT_DT"].notna(), [ID, "REPORT_DT"]].drop_duplicates()
reports = reports.loc[reports[ID].isin(surv.index)].copy()
reports["adt"] = reports[ID].map(stage1_anchor)
reports["d"] = (reports["REPORT_DT"] - reports["adt"]).dt.days


def pick_report(rule: str) -> pd.Series:
    r = reports
    if rule == "first_report":
        return r.groupby(ID)["REPORT_DT"].min()
    if rule == "first_report_on_or_after_adt":
        return r.loc[r["d"] >= 0].groupby(ID)["REPORT_DT"].min()
    if rule == "closest_report_to_adt":
        r = r.assign(absd=r["d"].abs()).sort_values([ID, "absd", "REPORT_DT"], kind="mergesort")
        return r.groupby(ID)["REPORT_DT"].first()
    raise ValueError(rule)


REPORT_RULES = ("first_report", "first_report_on_or_after_adt", "closest_report_to_adt")
s1_event = pd.to_numeric(surv[EVENT_COL], errors="coerce").fillna(0).eq(1)
s1_followup_end = surv["LAST_CONTACT_DATE"].where(~s1_event, surv[EVENT_DATE_COL])


def report_design(rule: str):
    index_date = pick_report(rule).reindex(surv.index)
    dur = (s1_followup_end - index_date).dt.days
    reasons = ladder(surv.index, [
        ("no somatic rows", pd.Series(~surv.index.isin(somatic[ID]), index=surv.index)),
        ("no report date", pd.Series(~surv.index.isin(reports[ID]), index=surv.index)),
        ("no report meeting the rule", index_date.isna()),
        (f"{ENDPOINT} on/before report", s1_event & ~(dur > 0)),
        ("last contact on/before report", ~(dur > 0)),
    ]).fillna("included")
    return index_date, reasons


report_results = {rule: report_design(rule) for rule in REPORT_RULES}
display(pd.concat({rule: reason_table(r)["n"] for rule, (_, r) in report_results.items()}, axis=1)
        .fillna(0).astype(int))

In [ ]:
def report_availability(rule: str, ids: pd.Index) -> pd.DataFrame:
    index_date, reasons = report_results[rule]
    inc = ids[reasons.reindex(ids).eq("included").values]
    idx = index_date.reindex(inc)
    gl = gleason_status_at(idx)
    out = pd.DataFrame(index=inc)
    out[f"{ENDPOINT} event after report"] = s1_event.reindex(inc).values
    out["on ADT at report (ADT start <= report)"] = (stage1_anchor.reindex(inc) <= idx).values
    out["Gleason available by report"] = gl.eq("available by index").values
    out["Gleason scored by report, documented later"] = gl.eq("scored by index, documented only after index").values
    out = out.join(pre_index_lab_flags(idx).drop(columns="n labs before index"))
    out["Gleason + somatic + PSA before report"] = out["Gleason available by report"] & out["PSA before index"]
    return out


summary = {}
for rule in REPORT_RULES:
    for layer, ids in (("stage1", surv.index), ("model", model_ids)):
        flags = report_availability(rule, ids)
        t = flag_table(flags)
        summary[(rule, layer)] = t["n"].astype(str) + " (" + t["pct"].astype(str) + "%)"
        summary[(rule, layer)].loc["n included"] = f"{len(flags):,}"
display(pd.DataFrame(summary).loc[lambda d: ["n included"] + [i for i in d.index if i != "n included"]])
print("Percentages are of included patients under each rule.")

for rule in REPORT_RULES:
    idx, reasons = report_results[rule]
    inc = reasons.eq("included")
    print(f"{rule}: report − ADT start (days) among included:",
          days_summary((idx[inc] - stage1_anchor[inc]).dt.days).to_dict())

### 6b. Side-by-side: ADT-start vs report-date designs

`n` is the analyzable cohort; `events` counts the endpoint after the index. For the ADT design the
Gleason/somatic columns use the available-case rules from §3b/§4b (pipeline collection-date rule).

In [ ]:
rows = [{
    "design": "ADT start (modeling cohort)",
    "n": len(model_ids),
    "events": int(base[EVENT_COL].sum()),
    "Gleason available": int(g_ok.sum()),
    "somatic available": int(s_ok.sum()),
    "Gleason + somatic": int((g_ok & s_ok).sum()),
    "PSA before index": int(lab_layers["model"]["PSA before index"].sum()),
}]
for rule in REPORT_RULES:
    flags = report_availability(rule, surv.index)
    rows.append({
        "design": f"report date: {rule} (Stage-1 base)",
        "n": len(flags),
        "events": int(flags[f"{ENDPOINT} event after report"].sum()),
        "Gleason available": int(flags["Gleason available by report"].sum()),
        "somatic available": len(flags),
        "Gleason + somatic": int(flags["Gleason available by report"].sum()),
        "PSA before index": int(flags["PSA before index"].sum()),
    })
display(pd.DataFrame(rows).set_index("design"))